In [1]:
import pandas as pd
from metagpt.tools.libs.data_preprocess import LabelEncode, OrdinalEncode

# Load the datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display data types per column
print("Train Data Types:")
print(train_df.dtypes)
print("\nTest Data Types:")
print(test_df.dtypes)

# Identify categorical columns
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()

# Initialize encoders
label_encoder = LabelEncode(features=categorical_cols)
ordinal_encoder = OrdinalEncode(features=categorical_cols)

# Fit and transform train data
train_encoded = train_df.copy()
train_encoded = label_encoder.fit_transform(train_encoded)
train_encoded = ordinal_encoder.fit_transform(train_encoded)

# Transform test data using the same encoders
test_encoded = test_df.copy()
test_encoded = label_encoder.transform(test_encoded)
test_encoded = ordinal_encoder.transform(test_encoded)

# Display the first few rows of the processed data
print("\nProcessed Train Data:")
print(train_encoded.head())
print("\nProcessed Test Data:")
print(test_encoded.head())


2025-09-15 07:17:06.559 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Train Data Types:
id                                        int64
no_of_adults                              int64
no_of_children                            int64
no_of_weekend_nights                      int64
no_of_week_nights                         int64
type_of_meal_plan                         int64
required_car_parking_space                int64
room_type_reserved                        int64
lead_time                                 int64
arrival_year                              int64
arrival_month                             int64
arrival_date                              int64
market_segment_type                       int64
repeated_guest                            int64
no_of_previous_cancellations              int64
no_of_previous_bookings_not_canceled      int64
avg_price_per_room                      float64
no_of_special_requests                    int64
booking_status                            int64
dtype: object

Test Data Types:
id                                    

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the processed train data
column_info_train = get_column_info(train_encoded)
print("Column Information for Train Data:")
print(column_info_train)

# Check column information for the processed test data
column_info_test = get_column_info(test_encoded)
print("\nColumn Information for Test Data:")
print(column_info_test)


Column Information for Train Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}

Column Information for Test Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}


In [3]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], early_stopping_rounds=50, verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)


TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [4]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'early_stopping_round': 50,
    'eval_metric': 'auc'
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], valid_names=['val'], early_stopping_rounds=50, verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [5]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'early_stopping_round': 50,
    'eval_metric': 'auc'
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], valid_names=['val'], verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)

TypeError: train() got an unexpected keyword argument 'verbose_eval'

In [6]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the processed train data
column_info_train = get_column_info(train_encoded)
print("Column Information for Train Data:")
print(column_info_train)

# Check column information for the processed test data
column_info_test = get_column_info(test_encoded)
print("\nColumn Information for Test Data:")
print(column_info_test)


Column Information for Train Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}

Column Information for Test Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}


In [7]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], early_stopping_rounds=50, verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)


TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [8]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'early_stopping_round': 50,
    'eval_metric': 'auc'
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], valid_names=['val'], early_stopping_rounds=50, verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [9]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'early_stopping_round': 50,
    'eval_metric': 'auc'
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], valid_names=['val'], verbose_eval=50)

# Predict on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Predict on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)

TypeError: train() got an unexpected keyword argument 'verbose_eval'

In [10]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the processed train data
column_info_train = get_column_info(train_encoded)
print("Column Information for Train Data:")
print(column_info_train)

# Check column information for the processed test data
column_info_test = get_column_info(test_encoded)
print("\nColumn Information for Test Data:")
print(column_info_test)


Column Information for Train Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}

Column Information for Test Data:
{'Category': [], 'Numeric': ['id', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status'], 'Datetime': [], 'Others': []}


In [11]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define features and target
features = train_encoded.columns.drop('booking_status')
target = 'booking_status'

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_encoded[features], train_encoded[target], test_size=0.2, random_state=42)

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Set hyperparameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the LightGBM model
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[val_data], early_stopping_rounds=50, verbose_eval=50)

# Make predictions on the validation set
y_val_pred = bst.predict(X_val, num_iteration=bst.best_iteration)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_val, y_val_pred)
print(f"Validation AUC-ROC: {auc_roc:.4f}")

# Make predictions on the test set
test_predictions = bst.predict(test_encoded[features], num_iteration=bst.best_iteration)

# Save the predictions to a CSV file
test_encoded['predicted_probability'] = test_predictions
test_encoded[['id', 'predicted_probability']].to_csv('test_predictions.csv', index=False)


TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'